# Death Count Extraction — Seq2Seq Rare-Bin Improvement Study

Training-side interventions on Flan-T5-Large to improve performance on rare death-count bins (3–5 and 6+).

| Strategy | Description |
|---|---|
| S0 | Baseline — standard fine-tuning, natural data distribution (results loaded) |
| S1 | WeightedRandomSampler — oversample rare-bin examples during training |
| S2 | Loss weighting — upweight rare-bin examples in cross-entropy |
| S3 | Targeted examples — oversample multi-group arithmetic and claimed-death cases |

**Protocol:** evaluate on validation set after each run to check for regressions on zero/low-count bins before comparing on the test set.

See `papers/death-counts/notes/rare-bin-attack-plan.md` for full study design.

## 1. Setup

### 1.1 Colab Setup

Mount Google Drive, clone the repo, install dependencies, and add the utils directory to `sys.path`.

In [ ]:
# 1) Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# 2) Clone or update the repo
BRANCH = "main"
!rm -rf /content/code-satp
!git clone -b $BRANCH --depth 1 https://github.com/eteitelbaum/code-satp.git /content/code-satp

# 3) Install dependencies
%pip install -qU pip setuptools wheel
%pip install -q -r /content/code-satp/models/count-models/requirements.txt

# 4) Paths and sys.path
import pathlib, sys
pathlib.Path("/content/drive/MyDrive/colab/satp-results/death-counts-rare-bin").mkdir(parents=True, exist_ok=True)
sys.path.append("/content/code-satp/models/count-models")

# 5) GPU check
import torch
print('=' * 60)
print('SETUP COMPLETE')
print('=' * 60)
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU — training will be very slow.')

TASK_NAME = "death-counts-rare-bin-seq2seq"
SEED = 42

### 1.2 Imports

Import all required libraries and set random seeds for reproducibility.

In [ ]:
import os, gc, json, re, warnings
import numpy as np
import pandas as pd
from pathlib import Path

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    EarlyStoppingCallback,
)

from utils import (
    compute_metrics, print_metrics,
    prepare_seq2seq_data,
    tokenize_seq2seq,
    extract_number,
    create_seq2seq_training_args,
    cleanup_model,
    compute_bin_weights,
    WeightedDataCollatorForSeq2Seq,
    WeightedSeq2SeqTrainer,
    LossWeightedSeq2SeqTrainer,
)

warnings.filterwarnings('ignore')
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('Libraries loaded.')

### 1.3 Config

Set model ID, training hyperparameters, and output directory paths.

In [ ]:
MODEL_ID   = "google/flan-t5-large"
BATCH_SIZE = 8       # reduce to 4 if OOM on T4
MAX_EPOCHS = 10
LR         = 3e-5

# All rare-bin seq2seq outputs (checkpoints + results) live on Drive,
# in a directory separate from the original death-counts experiment.
DRIVE_DIR   = Path('/content/drive/MyDrive/colab/satp-results/death-counts-seq2seq-rare-bin')
CKPT_DIR    = DRIVE_DIR / 'checkpoints'
NEW_RESULTS = DRIVE_DIR / 'results'
for _d in [DRIVE_DIR, CKPT_DIR, NEW_RESULTS]:
    _d.mkdir(parents=True, exist_ok=True)

# Original experiment results (S0 baseline) — read-only
S0_RESULTS = Path('/content/drive/MyDrive/colab/satp-results/death-counts')

print(f'Model:        {MODEL_ID}')
print(f'Checkpoints:  {CKPT_DIR}')
print(f'Results:      {NEW_RESULTS}')
print(f'S0 baseline:  {S0_RESULTS}')

## 2. Data

Load and reconstruct the three splits. The train split is derived by excluding val and test incident IDs from the full filtered dataset — the same procedure used in the original seq2seq experiment (SEED=42, stratified by bin).

### 2.1 Load and Split Data

In [ ]:
candidate_dirs = [
    Path('/content/code-satp/models/count-models/data'),
]
data_dir = next((d for d in candidate_dirs if (d / 'test.csv').exists()), None)
if data_dir is None:
    raise FileNotFoundError('Could not locate val.csv / test.csv')

val_df  = pd.read_csv(data_dir / 'val.csv')
test_df = pd.read_csv(data_dir / 'test.csv')

# Reconstruct training split. The original seq2seq notebook loaded satp_clean.csv
# and filtered to first_action in [Armed Assault, Bombing] before splitting 60/20/20.
# We replicate that filter here so the training distribution matches.
full_df = pd.read_csv('/content/code-satp/data/satp_clean.csv')
df_filtered = full_df[full_df['first_action'].isin(['Armed Assault', 'Bombing'])].copy()

exclude_ids = set(val_df['incident_number'].tolist()) | set(test_df['incident_number'].tolist())
train_df = df_filtered[~df_filtered['incident_number'].isin(exclude_ids)].copy()

# Bin labels
def assign_bin(n):
    if n == 0:   return '0'
    if n == 1:   return '1'
    if n == 2:   return '2'
    if n <= 5:   return '3-5'
    return '6+'

for df in [train_df, val_df, test_df]:
    df['bin'] = df['total_fatalities'].apply(assign_bin)

print(f'Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}')
print()
print('Train bin distribution:')
print(train_df['bin'].value_counts().sort_index())

## 3. Helpers

### 3.1 Define Helpers

In [ ]:
def evaluate_model(trainer, tokenizer, df, split_name, strategy_name):
    """Run prediction on a split and return (metrics, predictions_df)."""
    data    = prepare_seq2seq_data(df, model_type='flan-t5')
    dataset = Dataset.from_dict(data)
    tok_ds  = dataset.map(
        lambda x: tokenize_seq2seq(x, tokenizer, max_input_length=512),
        batched=True, remove_columns=dataset.column_names
    )
    tok_ds.set_format('torch')

    preds = trainer.predict(tok_ds)
    decoded = tokenizer.batch_decode(
        np.where(preds.predictions == -100, tokenizer.pad_token_id, preds.predictions),
        skip_special_tokens=True,
    )
    parsed = [extract_number(s) if s.strip() else 0 for s in decoded]
    ok     = [bool(s.strip()) for s in decoded]
    true   = df['total_fatalities'].values

    metrics = compute_metrics(parsed, true, ok)

    res = df[['incident_number', 'incident_summary', 'total_fatalities', 'bin']].copy()
    res['prediction'] = parsed
    res['raw_output'] = decoded
    res['error'] = res['prediction'] - res['total_fatalities']

    out_stem = f'{strategy_name}_{split_name}'
    res.to_csv(NEW_RESULTS / f'{out_stem}_predictions.csv', index=False)
    with open(NEW_RESULTS / f'{out_stem}_metrics.json', 'w') as f:
        json.dump(metrics, f, indent=2)

    print(f'\n=== {strategy_name} | {split_name} ===')
    print_bin_table(metrics)
    return metrics, res


def print_bin_table(metrics):
    """Print a compact bin-level table from a metrics dict."""
    print(f'  {"Bin":<6}  {"N":>4}  {"MAE":>6}  {"Exact%":>7}  {"±1%":>6}')
    for b in ['0', '1', '2', '3-5', '6+']:
        bm = metrics['bins'].get(b, {})
        if not bm:
            continue
        print(f'  {b:<6}  {bm["n"]:>4}  {bm["mae"]:>6.3f}  '
              f'{bm["exact_match"]*100:>6.1f}%  {bm["within_1"]*100:>5.1f}%')
    om = metrics['overall']
    print(f'  {"Overall":<6}  {"":>4}  {om["mae"]:>6.3f}  '
          f'{om["exact_match"]*100:>6.1f}%  {om["within_1"]*100:>5.1f}%')


def already_done(strategy_name, split_name):
    return (NEW_RESULTS / f'{strategy_name}_{split_name}_metrics.json').exists()


print('Helpers defined.')

## 4. S0 — Baseline

Load Flan-T5-Large results from the original seq2seq experiment. No retraining needed.

### 4.1 Load Baseline Results

In [ ]:
s0_metrics_path = S0_RESULTS / 'death_counts_flan-t5-large_metrics.json'
if s0_metrics_path.exists():
    with open(s0_metrics_path) as f:
        s0_metrics = json.load(f)
    print('S0 baseline (Flan-T5-Large):')
    print_bin_table(s0_metrics)
else:
    print(f'WARNING: S0 metrics not found at {s0_metrics_path}')
    print('Run the original death-count-extraction-seq2seq.ipynb first.')

### 4.2 Tokenizer Diagnostic

Verify that `text_target=` produces correct label encodings. Also tests `as_target_tokenizer()` for comparison if available in this transformers version.

In [ ]:
from transformers import AutoTokenizer

_tok = AutoTokenizer.from_pretrained(MODEL_ID)
sample_targets = ['0', '1', '2', '4', '12', '25']

print("Pad token ID:", _tok.pad_token_id)
print("EOS token ID:", _tok.eos_token_id)
print()

# Current approach: text_target=
enc = _tok(text_target=sample_targets, max_length=10, truncation=True, padding='max_length')
labels = [[(l if l != _tok.pad_token_id else -100) for l in seq] for seq in enc['input_ids']]

print("text_target= approach:")
for tgt, lbl in zip(sample_targets, labels):
    non_pad = [l for l in lbl if l != -100]
    decoded = _tok.decode(non_pad, skip_special_tokens=True)
    print(f"  target={tgt!r:>4}  tokens={non_pad}  decoded={decoded!r}")

# Old approach: as_target_tokenizer()
print()
try:
    with _tok.as_target_tokenizer():
        enc_old = _tok(sample_targets, max_length=10, truncation=True, padding='max_length')
    labels_old = [[(l if l != _tok.pad_token_id else -100) for l in seq] for seq in enc_old['input_ids']]
    print("as_target_tokenizer() approach:")
    for tgt, lbl in zip(sample_targets, labels_old):
        non_pad = [l for l in lbl if l != -100]
        decoded = _tok.decode(non_pad, skip_special_tokens=True)
        print(f"  target={tgt!r:>4}  tokens={non_pad}  decoded={decoded!r}")
    match = all(a == b for a, b in zip(labels, labels_old))
    print(f"\nApproaches produce identical labels: {match}")
except AttributeError:
    print("as_target_tokenizer() not available in this transformers version.")
    print("Cannot compare — proceeding with text_target= only.")

del _tok

### 4.3 Reproduce Baseline

Train Flan-T5-Large from scratch with standard `Seq2SeqTrainer` and no weighting. If this matches the pre-computed S0 results, the training setup is correct. If it also collapses, the issue is in the training setup, not the weighting strategies.

In [ ]:
tokenizer_s0r = AutoTokenizer.from_pretrained(MODEL_ID)

train_data_s0r = prepare_seq2seq_data(train_df, model_type='flan-t5')
val_data_s0r   = prepare_seq2seq_data(val_df,   model_type='flan-t5')

train_tok_s0r = Dataset.from_dict(train_data_s0r).map(
    lambda x: tokenize_seq2seq(x, tokenizer_s0r, max_input_length=512),
    batched=True, remove_columns=['input', 'target']
)
val_tok_s0r = Dataset.from_dict(val_data_s0r).map(
    lambda x: tokenize_seq2seq(x, tokenizer_s0r, max_input_length=512),
    batched=True, remove_columns=['input', 'target']
)
train_tok_s0r.set_format('torch')
val_tok_s0r.set_format('torch')

if not already_done('s0_reproduced', 'val'):
    model_s0r   = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)
    collator_s0r = DataCollatorForSeq2Seq(tokenizer=tokenizer_s0r, model=model_s0r, padding=True)
    args_s0r     = create_seq2seq_training_args(
        output_dir=str(CKPT_DIR / 's0_reproduced'),
        batch_size=BATCH_SIZE, learning_rate=LR, num_epochs=MAX_EPOCHS, seed=SEED
    )
    trainer_s0r = Seq2SeqTrainer(
        model=model_s0r,
        args=args_s0r,
        train_dataset=train_tok_s0r,
        eval_dataset=val_tok_s0r,
        data_collator=collator_s0r,
        tokenizer=tokenizer_s0r,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )
    print('Training S0 reproduction...')
    trainer_s0r.train()
    print('Training complete.')
else:
    print('S0 reproduction already done — skipping training.')

### 4.4 Evaluate Reproduced Baseline

Compare against pre-computed S0 results. If they match, the current training setup is sound.

In [ ]:
s0r_val_metrics, s0r_val_preds = evaluate_model(
    trainer_s0r, tokenizer_s0r, val_df, 'val', 's0_reproduced'
)
s0r_test_metrics, s0r_test_preds = evaluate_model(
    trainer_s0r, tokenizer_s0r, test_df, 'test', 's0_reproduced'
)
cleanup_model(trainer_s0r, model_s0r)

## 5. S1 — WeightedRandomSampler

Oversample rare-bin training examples using `WeightedSeq2SeqTrainer`.
Bin weights are inverse-frequency, capped at 4× the median weight, then sqrt-scaled.

### 5.1 Data Preparation

In [ ]:
# Tokenize training and validation data
tokenizer_s1 = AutoTokenizer.from_pretrained(MODEL_ID)

train_data_s1 = prepare_seq2seq_data(train_df, model_type='flan-t5')
val_data_s1   = prepare_seq2seq_data(val_df,   model_type='flan-t5')

train_hf_s1 = Dataset.from_dict(train_data_s1)
val_hf_s1   = Dataset.from_dict(val_data_s1)

train_tok_s1 = train_hf_s1.map(
    lambda x: tokenize_seq2seq(x, tokenizer_s1, max_input_length=512),
    batched=True, remove_columns=train_hf_s1.column_names
)
val_tok_s1 = val_hf_s1.map(
    lambda x: tokenize_seq2seq(x, tokenizer_s1, max_input_length=512),
    batched=True, remove_columns=val_hf_s1.column_names
)
train_tok_s1.set_format('torch')
val_tok_s1.set_format('torch')

# Compute per-example sampling weights
sample_weights_s1 = compute_bin_weights(train_df['total_fatalities'].tolist())

print(f'Training examples: {len(train_tok_s1)}')
for b in ['0', '1', '2', '3-5', '6+']:
    sub = train_df[train_df['bin'] == b]
    w = sample_weights_s1[train_df[train_df['bin'] == b].index[0]] if len(sub) > 0 else 0
    print(f'  Bin {b}: n={len(sub):4d}  weight={w:.3f}')

### 5.2 Training

In [ ]:
if not already_done('s1_weighted_sampler', 'val'):
    model_s1 = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)
    collator_s1 = DataCollatorForSeq2Seq(tokenizer=tokenizer_s1, model=model_s1, padding=True)

    args_s1 = create_seq2seq_training_args(
        output_dir=str(CKPT_DIR / 's1'),
        batch_size=BATCH_SIZE, learning_rate=LR, num_epochs=MAX_EPOCHS, seed=SEED
    )

    trainer_s1 = WeightedSeq2SeqTrainer(
        sample_weights=sample_weights_s1,
        model=model_s1,
        args=args_s1,
        train_dataset=train_tok_s1,
        eval_dataset=val_tok_s1,
        data_collator=collator_s1,
        tokenizer=tokenizer_s1,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    print('Training S1 (WeightedRandomSampler)...')
    trainer_s1.train()
    print('Training complete.')
else:
    print('S1 val results found — skipping training. Load from disk to re-evaluate.')

### 5.3 Validation Evaluation

In [ ]:
# Evaluate on val first; proceed to test only after checking bin-0 doesn't degrade
s1_val_metrics, s1_val_preds = evaluate_model(
    trainer_s1, tokenizer_s1, val_df, 'val', 's1_weighted_sampler'
)

# Regression check: bin-0 exact match should not fall significantly below S0
s0_bin0 = s0_metrics['bins']['0']['exact_match'] if 's0_metrics' in dir() else None
s1_bin0 = s1_val_metrics['bins']['0']['exact_match']
if s0_bin0:
    delta = s1_bin0 - s0_bin0
    status = 'OK' if delta >= -0.02 else 'WARNING: regression on bin-0'
    print(f'\nBin-0 regression check: S0={s0_bin0:.3f}  S1={s1_bin0:.3f}  delta={delta:+.3f}  {status}')

### 5.4 Test Evaluation

In [ ]:
# Only run test set after confirming no regression
s1_test_metrics, s1_test_preds = evaluate_model(
    trainer_s1, tokenizer_s1, test_df, 'test', 's1_weighted_sampler'
)
cleanup_model(trainer_s1, model_s1)

## 6. S2 — Loss Weighting

Upweight rare-bin examples in the cross-entropy loss using `LossWeightedSeq2SeqTrainer`.
Weights are the same bin-inverse-frequency values as S1 but applied at the loss level
rather than the sampling level.

### 6.1 Data Preparation

In [ ]:
tokenizer_s2 = AutoTokenizer.from_pretrained(MODEL_ID)

train_data_s2 = prepare_seq2seq_data(train_df, model_type='flan-t5')
val_data_s2   = prepare_seq2seq_data(val_df,   model_type='flan-t5')

train_hf_s2 = Dataset.from_dict(train_data_s2)
val_hf_s2   = Dataset.from_dict(val_data_s2)

train_tok_s2 = train_hf_s2.map(
    lambda x: tokenize_seq2seq(x, tokenizer_s2, max_input_length=512),
    batched=True, remove_columns=train_hf_s2.column_names
)
val_tok_s2 = val_hf_s2.map(
    lambda x: tokenize_seq2seq(x, tokenizer_s2, max_input_length=512),
    batched=True, remove_columns=val_hf_s2.column_names
)

# Add sample_weight column so LossWeightedSeq2SeqTrainer can read it from each batch
sample_weights_s2 = compute_bin_weights(train_df['total_fatalities'].tolist())
train_tok_s2 = train_tok_s2.add_column('sample_weight', sample_weights_s2)
train_tok_s2.set_format('torch')
val_tok_s2.set_format('torch')

print(f'Training examples: {len(train_tok_s2)}')
print('sample_weight column added to training dataset')

### 6.2 Training

In [ ]:
if not already_done('s2_loss_weighted', 'val'):
    model_s2 = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)
    collator_s2 = WeightedDataCollatorForSeq2Seq(tokenizer=tokenizer_s2, model=model_s2, padding=True)

    args_s2 = create_seq2seq_training_args(
        output_dir=str(CKPT_DIR / 's2'),
        batch_size=BATCH_SIZE, learning_rate=LR, num_epochs=MAX_EPOCHS, seed=SEED
    )

    trainer_s2 = LossWeightedSeq2SeqTrainer(
        model=model_s2,
        args=args_s2,
        train_dataset=train_tok_s2,
        eval_dataset=val_tok_s2,
        data_collator=collator_s2,
        tokenizer=tokenizer_s2,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    print('Training S2 (loss weighting)...')
    trainer_s2.train()
    print('Training complete.')
else:
    print('S2 val results found — skipping training.')

### 6.3 Evaluation

In [ ]:
s2_val_metrics, s2_val_preds = evaluate_model(
    trainer_s2, tokenizer_s2, val_df, "val", "s2_loss_weighted"
)

### 6.4 Regression Check

Inspect val results above before running the test set. Confirm bin-0 exact match does not degrade vs S0 (98.1%) before proceeding.

In [ ]:
s2_test_metrics, s2_test_preds = evaluate_model(
    trainer_s2, tokenizer_s2, test_df, "test", "s2_loss_weighted"
)
cleanup_model(trainer_s2, model_s2)

## 7. S3 — Targeted Training Examples

Identify training examples that match the two structural failure modes found in the
diagnostic analysis, then oversample them 3× in the training data:

1. **Multi-group arithmetic** — narratives listing deaths across two or more named groups
   (e.g. "Three Maoists and a civilian were killed"). T5-Large's unique errors were on
   exactly these structures.

2. **Claimed / bodies not recovered** — attacker deaths reported with uncertainty
   ("claimed", "bodies taken away", "not recovered"). Universal LLM and seq2seq failure.

The training labels for both types already encode the maximalist count protocol — the
model just needs more exposure to these patterns.

### 7.1 Find Examples

In [ ]:
# Regex patterns for the two failure modes
MULTI_GROUP_RE = re.compile(
    r'\b(and|along with)\b.{1,40}\b(killed|dead|shot|gunned down)\b'
    r'|\b(killed|dead|shot|gunned down)\b.{1,40}\b(and|along with)\b',
    re.IGNORECASE
)
CLAIMED_RE = re.compile(
    r'\b(claimed|bodies taken|bodies not recovered|could not be recovered'
    r'|not recovered from|taken away by)\b',
    re.IGNORECASE
)

train_df['is_multigroup'] = train_df['incident_summary'].str.contains(
    MULTI_GROUP_RE, regex=True
)
train_df['is_claimed']    = train_df['incident_summary'].str.contains(
    CLAIMED_RE, regex=True
)
train_df['is_targeted']   = train_df['is_multigroup'] | train_df['is_claimed']

targeted = train_df[train_df['is_targeted'] & (train_df['total_fatalities'] > 0)]
print(f'Multi-group matches:  {train_df["is_multigroup"].sum()}')
print(f'Claimed/bodies matches: {train_df["is_claimed"].sum()}')
print(f'Targeted (either, fatalities>0): {len(targeted)}')
print()
print('Targeted bin distribution:')
print(targeted['bin'].value_counts().sort_index())

### 7.2 Build Augmented Dataset

In [ ]:
OVERSAMPLE_FACTOR = 3  # add 2 extra copies of each targeted example

augmented_df = pd.concat(
    [train_df] + [targeted] * (OVERSAMPLE_FACTOR - 1),
    ignore_index=True
)

print(f'Original training set:  {len(train_df):,}')
print(f'Augmented training set: {len(augmented_df):,}  '
      f'(+{len(augmented_df) - len(train_df):,} targeted examples)')
print()
print('Augmented bin distribution:')
print(augmented_df['bin'].value_counts().sort_index())

### 7.3 Training

In [ ]:
tokenizer_s3 = AutoTokenizer.from_pretrained(MODEL_ID)

train_data_s3 = prepare_seq2seq_data(augmented_df, model_type='flan-t5')
val_data_s3   = prepare_seq2seq_data(val_df,        model_type='flan-t5')

train_hf_s3 = Dataset.from_dict(train_data_s3)
val_hf_s3   = Dataset.from_dict(val_data_s3)

train_tok_s3 = train_hf_s3.map(
    lambda x: tokenize_seq2seq(x, tokenizer_s3, max_input_length=512),
    batched=True, remove_columns=train_hf_s3.column_names
)
val_tok_s3 = val_hf_s3.map(
    lambda x: tokenize_seq2seq(x, tokenizer_s3, max_input_length=512),
    batched=True, remove_columns=val_hf_s3.column_names
)
train_tok_s3.set_format('torch')
val_tok_s3.set_format('torch')

if not already_done('s3_targeted_examples', 'val'):
    model_s3 = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)
    collator_s3 = DataCollatorForSeq2Seq(tokenizer=tokenizer_s3, model=model_s3, padding=True)

    args_s3 = create_seq2seq_training_args(
        output_dir=str(CKPT_DIR / 's3'),
        batch_size=BATCH_SIZE, learning_rate=LR, num_epochs=MAX_EPOCHS, seed=SEED
    )

    trainer_s3 = Seq2SeqTrainer(
        model=model_s3,
        args=args_s3,
        train_dataset=train_tok_s3,
        eval_dataset=val_tok_s3,
        data_collator=collator_s3,
        tokenizer=tokenizer_s3,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    print('Training S3 (targeted examples)...')
    trainer_s3.train()
    print('Training complete.')
else:
    print('S3 val results found — skipping training.')

### 7.4 Evaluation

In [ ]:
s3_val_metrics, s3_val_preds = evaluate_model(
    trainer_s3, tokenizer_s3, val_df, "val", "s3_targeted_examples"
)

### 7.5 Regression Check

Inspect val results above before running the test set. Confirm bin-0 exact match does not degrade vs S0 (98.1%) before proceeding.

In [ ]:
s3_test_metrics, s3_test_preds = evaluate_model(
    trainer_s3, tokenizer_s3, test_df, "test", "s3_targeted_examples"
)
cleanup_model(trainer_s3, model_s3)

## 8. S6 — Inference-Time Few-Shot on Best Fine-Tuned Model

Test whether prepending few-shot examples to the inference prompt adds further gains
on top of the best S1–S3 model. The fine-tuned model was trained on plain inputs, so
this is out-of-distribution at inference time — the result is genuinely uncertain.

Uses the same hard-case examples as the LLM track (L3) for direct comparability.
Run on the validation set first; if it helps, run on test.

### 8.1 Setup

In [ ]:
# ── S6: few-shot prompting on the best fine-tuned model ─────────────────────
# Set BEST_TRAINER and BEST_TOKENIZER to whichever S1/S2/S3 model performed best
# on the validation set. Change these assignments after reviewing val results above.
BEST_TRAINER   = trainer_s3      # <-- update if S1 or S2 was better
BEST_TOKENIZER = tokenizer_s3    # <-- update to match

# Hard-case few-shot examples (same as LLM track L3, from training set)
FEW_SHOT_EXAMPLES = [
    # Multi-group arithmetic (training ID 312160501)
    ("Three Maoists and a civilian were killed during an encounter at "
     "Bhejji locality in the Dantewada District.", "4"),
    # Claimed attacker deaths, bodies carried away (training ID 201130801)
    ("Police claimed to have killed six cadres of the CPI-Maoist in an "
     "encounter at Bangudwa Naktaia hills in the Gaya District. The Deputy "
     "Superintendent of Police said that dead bodies of the slain Maoists "
     "could not be recovered from the encounter site as these were taken "
     "away by their colleagues.", "6"),
    # Succumbed to injuries (training ID 303031602)
    ("Three troopers of CoBRA were killed and at least 15 others were "
     "injured in an encounter with CPI-Maoist cadres in Sukma District. "
     "Officials said while two Commandos had succumbed to bullet injuries "
     "on March 3, their colleague died on March 4. At least 15 others "
     "were injured.", "3"),
    # Injuries don't count; claimed deaths without body recovery do (training ID 312081501)
    ("Five security personnel, including two STF troopers, were injured "
     "when CPI-Maoist cadres ambushed a team of SFs in Sukma District. "
     "Police also claimed to have gunned down at least 15 Maoists in the "
     "encounter although no bodies were recovered from the spot.", "15"),
]

INSTR = "How many people were killed? Answer with only a number."
shot_block = "\n\n".join(f"Text: {ex}\nAnswer: {ans}" for ex, ans in FEW_SHOT_EXAMPLES)

def make_fewshot_input_t5(text):
    return f"{INSTR}\n\n{shot_block}\n\nText: {text}\nAnswer:"

# Evaluate on val set using few-shot prompts
def evaluate_fewshot(model, tokenizer, df, split_name, strategy_name):
    """Encode few-shot prompts directly and run greedy decoding."""
    prompts = [make_fewshot_input_t5(t) for t in df['incident_summary'].tolist()]
    device  = next(model.parameters()).device

    all_preds = []
    BS = 8
    for i in range(0, len(prompts), BS):
        batch = prompts[i:i+BS]
        enc   = tokenizer(batch, return_tensors='pt', padding=True,
                          truncation=True, max_length=512).to(device)
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=8)
        decoded = tokenizer.batch_decode(out, skip_special_tokens=True)
        all_preds.extend(decoded)

    parsed = [extract_number(s) if s.strip() else 0 for s in all_preds]
    ok     = [bool(s.strip()) for s in all_preds]
    true   = df['total_fatalities'].values
    metrics = compute_metrics(parsed, true, ok)

    res = df[['incident_number', 'incident_summary', 'total_fatalities', 'bin']].copy()
    res['prediction'] = parsed
    res['raw_output'] = all_preds
    res['error'] = res['prediction'] - res['total_fatalities']

    out_stem = f'{strategy_name}_{split_name}'
    res.to_csv(NEW_RESULTS / f'{out_stem}_predictions.csv', index=False)
    with open(NEW_RESULTS / f'{out_stem}_metrics.json', 'w') as fh:
        json.dump(metrics, fh, indent=2)

    print(f'\n=== {strategy_name} | {split_name} ===')
    print_bin_table(metrics)
    return metrics, res

s6_val_metrics, s6_val_preds = evaluate_fewshot(
    BEST_TRAINER.model, BEST_TOKENIZER, val_df, 'val', 's6_fewshot'
)

### 8.2 Test Evaluation

In [ ]:
# Run on test set only if val shows improvement over the best S1–S3 val result
# Compare s6_val_metrics vs the best of s1/s2/s3 val metrics before proceeding
s6_test_metrics, s6_test_preds = evaluate_fewshot(
    BEST_TRAINER.model, BEST_TOKENIZER, test_df, 'test', 's6_fewshot'
)

## 9. Results Table

Compile bin-level metrics across all strategies. The T5-XL-QLoRA row is loaded from
the original experiment as an upper-bound reference.

### 9.1 Compile Results

In [ ]:
def load_metrics(path):
    if Path(path).exists():
        with open(path) as f:
            return json.load(f)
    return None

xl_metrics = load_metrics(S0_RESULTS / 'death_counts_flan-t5-xl-lora_metrics.json')
s0_m  = load_metrics(S0_RESULTS / 'death_counts_flan-t5-large_metrics.json')
s1_m  = load_metrics(NEW_RESULTS / 's1_weighted_sampler_test_metrics.json')
s2_m  = load_metrics(NEW_RESULTS / 's2_loss_weighted_test_metrics.json')
s3_m  = load_metrics(NEW_RESULTS / 's3_targeted_examples_test_metrics.json')

rows = [
    ('T5-XL-QLoRA (ceiling)',    xl_metrics),
    ('S0 — Baseline',            s0_m),
    ('S1 — Weighted sampling',   s1_m),
    ('S2 — Loss weighting',      s2_m),
    ('S3 — Targeted examples',   s3_m),
    ('S6 — Few-shot on best model', load_metrics(NEW_RESULTS / 's6_fewshot_test_metrics.json')),
]

print(f'{"Strategy":<30}  {"Overall":>7}  {"MAE_35":>7}  {"Exact35":>8}  {"MAE_6+":>7}  {"Exact6+":>8}')
print('-' * 80)
for name, m in rows:
    if m is None:
        print(f'{name:<30}  (not available)')
        continue
    mae_all = m['overall']['mae']
    b35 = m['bins'].get('3-5', {})
    b6  = m['bins'].get('6+',  {})
    mae35   = b35.get('mae', float('nan'))
    exact35 = b35.get('exact_match', float('nan')) * 100
    mae6    = b6.get('mae', float('nan'))
    exact6  = b6.get('exact_match', float('nan')) * 100
    print(f'{name:<30}  {mae_all:>7.3f}  {mae35:>7.3f}  {exact35:>7.1f}%  {mae6:>7.3f}  {exact6:>7.1f}%')

### 9.2 Save CSV

In [ ]:
# Save combined results as CSV for the paper
table_rows = []
for name, m in rows:
    if m is None:
        continue
    b35 = m['bins'].get('3-5', {})
    b6  = m['bins'].get('6+',  {})
    table_rows.append({
        'strategy':    name,
        'overall_mae': m['overall']['mae'],
        'overall_exact': m['overall']['exact_match'],
        'nonzero_mae': m['overall']['nonzero_mae'],
        'bin35_mae':   b35.get('mae'),
        'bin35_exact': b35.get('exact_match'),
        'bin6_mae':    b6.get('mae'),
        'bin6_exact':  b6.get('exact_match'),
    })

results_table = pd.DataFrame(table_rows)
out_path = NEW_RESULTS / 'seq2seq_rare_bin_results.csv'
results_table.to_csv(out_path, index=False)
print(f'Saved: {out_path}')
print(results_table.to_string(index=False))